In [1]:
storage_account = "straccajs"

# DEVELOP MODE - January only
raw_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/yellow/yellow_tripdata_2024-01.parquet"

# FULL RUN - uncomment for final demo
# raw_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/yellow/"

processed_path = f"abfss://processed@{storage_account}.dfs.core.windows.net/yellow/"
print(f"Reading from: {raw_path}")

StatementMeta(sparkpool1, 0, 2, Finished, Available, Finished, False)

Reading from: abfss://raw@straccajs.dfs.core.windows.net/yellow/yellow_tripdata_2024-01.parquet


This sets the input and output paths.

In [2]:
from pyspark.sql import functions as F

df_raw = spark.read.parquet(raw_path)
df_raw.cache()
total_raw = df_raw.count()
print(f"Raw row count: {total_raw:,}")
df_raw.printSchema()

StatementMeta(sparkpool1, 0, 3, Finished, Available, Finished, False)

Raw row count: 2,964,624
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [3]:
expected_columns = {
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "PULocationID", "DOLocationID",
    "fare_amount", "tip_amount", "total_amount", "payment_type"
}
actual = set(df_raw.columns)
missing = expected_columns - actual
if missing:
    raise ValueError(f"Schema validation failed - missing columns: {missing}")
print("Schema validation passed.")

StatementMeta(sparkpool1, 0, 4, Finished, Available, Finished, False)

Schema validation passed.


In [4]:
profile = df_raw.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("tpep_pickup_datetime").isNull(), 1).otherwise(0)).alias("null_pickup"),
    F.sum(F.when(F.col("fare_amount").isNull(), 1).otherwise(0)).alias("null_fare"),
    F.sum(F.when(F.col("fare_amount") < 0, 1).otherwise(0)).alias("negative_fare"),
    F.sum(F.when(F.col("trip_distance") == 0, 1).otherwise(0)).alias("zero_distance"),
    F.sum(F.when(F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"), 1).otherwise(0)).alias("dropoff_before_pickup"),
    F.sum(F.when(F.year("tpep_pickup_datetime") != 2024, 1).otherwise(0)).alias("wrong_year")
).collect()[0]

print("Quality profile:")
for k in profile.asDict():
    print(f"  {k}: {profile[k]:,}")

StatementMeta(sparkpool1, 0, 5, Finished, Available, Finished, False)

Quality profile:
  total_rows: 2,964,624
  null_pickup: 0
  null_fare: 0
  negative_fare: 37,448
  zero_distance: 60,371
  dropoff_before_pickup: 870
  wrong_year: 15


In [5]:
df_raw.write.mode("overwrite").parquet(processed_path)
print(f"Wrote raw -> processed: {total_raw:,} rows")
df_raw.unpersist()

StatementMeta(sparkpool1, 0, 6, Finished, Available, Finished, False)

Wrote raw -> processed: 2,964,624 rows


DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double]